# max-back-tied-half — ex3: verify per-position mass conservation across all tie patterns

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `max-back-tied-half`. Running the final beacon cell reports progress against the `Backprop: max_back with tied half-mass` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: max_back with tied half-mass` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`max-back-tied-half`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "max-back-tied-half"
DD_SUBTOPIC = "Backprop: max_back with tied half-mass"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## maximum_back — mass conservation across half-mass ties

Ex1 derived the half-mass tie rule; ex2 specialised it to `relu_back`. The deepening move tests a STRUCTURAL INVARIANT both earlier exercises rely on but never check directly: per-position grad mass is conserved.

```
for every position i:
    maximum_back0[i] + maximum_back1[i]  ==  grad_out[i]
```

Case-by-case proof:

| condition       | bool_sum_x       | bool_sum_y       | sum   |
|-----------------|------------------|------------------|-------|
| `x > y`         | 1.0              | 0.0              | 1.0   |
| `x < y`         | 0.0              | 1.0              | 1.0   |
| `x == y` (tie)  | 0.5              | 0.5              | 1.0   |

Multiply both sides by `grad_out[i]`: the sum of per-arg-position grads equals `grad_out[i]` at every position, with no leak and no double-count.

**Why this matters.** Mass conservation is the autograd correctness oracle for any back-fn that splits a single gradient across multiple parents. A naive implementation that returned `grad_out * (x >= y)` for BOTH back0 and back1 would double-count at the kink (sum = `2 * grad_out`). The half-mass rule is the unique split that conserves mass AND remains symmetric across the tie.

### Exercise 3 — verify per-position mass conservation across all tie patterns

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the mass-conservation invariant of half-mass tie-splitting: for every position i, maximum_back0[i] + maximum_back1[i] == grad_out[i], holds across x>y, x<y, and x==y cases.
> Keywords: mass-conservation, tie, invariant, back0, back1
> ```

**KCs targeted:** `mass-conservation-invariant`, `half-mass-symmetric-split`

Implement `maximum_back0(grad_out, out, x, y)`, `maximum_back1(grad_out, out, x, y)`, and a verifier `ex3_check_mass_conservation(x, y, grad_out)` that returns a dict summarising the invariant on a per-position basis.

Function 1+2 — the back-fns (no unbroadcast; assume `x.shape == y.shape == grad_out.shape`):
- `maximum_back0(grad_out, out, x, y) = grad_out * ((x > y) + 0.5 * (x == y))`.
- `maximum_back1(grad_out, out, x, y) = grad_out * ((y > x) + 0.5 * (x == y))`. Note the symmetric inequality flipped to `y > x`.

Function 3 — `ex3_check_mass_conservation(x, y, grad_out)`. Returns a dict with these keys:

- `'g0'`: the tensor `maximum_back0(grad_out, _, x, y)`.
- `'g1'`: the tensor `maximum_back1(grad_out, _, x, y)`.
- `'sum'`: the elementwise sum `g0 + g1`.
- `'conserves_mass'`: bool — True iff `t.allclose(g0 + g1, grad_out, atol=1e-6)`.
- `'n_ties'`: int — number of positions where `x == y`.
- `'n_x_wins'`: int — positions where `x > y`.
- `'n_y_wins'`: int — positions where `x < y`.

The verifier must work on tensors of any shape; the position counts are over `.numel()`.

In [ ]:
def maximum_back0(grad_out, out, x, y):
    bool_sum_x = (x > y).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype)
    return grad_out * bool_sum_x


def maximum_back1(grad_out, out, x, y):
    bool_sum_y = (y > x).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype)
    return grad_out * bool_sum_y


def ex3_check_mass_conservation(x, y, grad_out):
    g0 = maximum_back0(grad_out, None, x, y)
    g1 = maximum_back1(grad_out, None, x, y)
    s = g0 + g1
    return {
        'g0': g0,
        'g1': g1,
        'sum': s,
        'conserves_mass': bool(t.allclose(s, grad_out, atol=1e-6)),
        'n_ties': int((x == y).sum().item()),
        'n_x_wins': int((x > y).sum().item()),
        'n_y_wins': int((y > x).sum().item()),
    }


<details><summary>Solution</summary>

```python
def maximum_back0(grad_out, out, x, y):
    bool_sum_x = (x > y).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype)
    return grad_out * bool_sum_x


def maximum_back1(grad_out, out, x, y):
    bool_sum_y = (y > x).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype)
    return grad_out * bool_sum_y


def ex3_check_mass_conservation(x, y, grad_out):
    g0 = maximum_back0(grad_out, None, x, y)
    g1 = maximum_back1(grad_out, None, x, y)
    s = g0 + g1
    return {
        'g0': g0,
        'g1': g1,
        'sum': s,
        'conserves_mass': bool(t.allclose(s, grad_out, atol=1e-6)),
        'n_ties': int((x == y).sum().item()),
        'n_x_wins': int((x > y).sum().item()),
        'n_y_wins': int((y > x).sum().item()),
    }
```

**Mass conservation = correctness oracle.** Any back-fn that splits one upstream gradient across multiple parents must preserve total mass. If the sum exceeds `grad_out`, the loss surface is double-counted; if it falls short, gradient flow is lossy. The half-mass rule is the UNIQUE symmetric split that conserves mass at ties.

**Three exhaustive cases.** At any position one of: `x > y`, `x < y`, `x == y` holds. Verify the three cases produce sum=1 (times `grad_out`): (1+0)=1, (0+1)=1, (0.5+0.5)=1. There's no fourth case to worry about.

**Cast booleans before multiplication.** `(x > y)` is a `torch.bool` tensor; multiplying it directly works (auto-coerces) but adding `0.5 * (x == y)` to it requires explicit `.to(...)`. Cast to `grad_out.dtype` keeps the result dtype-correct and avoids a sneaky upcast to float64 when `grad_out` is float32.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()